## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-08-10-44-42 +0000,bbc,Government 'looking seriously' at banning unde...,https://www.bbc.co.uk/news/articles/cmzxjvpqrj...
1,2026-11-08-10-36-15 +0000,nypost,NYC ‘principal’ — who’s also a rapper — being ...,https://nypost.com/2026/11/08/us-news/nyc-doe-...
2,2026-11-08-10-30-00 +0000,wsj,Do You Know England’s Founding Father? It’s OK...,https://www.wsj.com/world/uk/do-you-know-engla...
3,2026-11-08-10-30-00 +0000,wsj,A Democratic Senate majority would be the broa...,https://www.wsj.com/economy/central-banking/th...
4,2026-11-08-10-30-00 +0000,wsj,Don’t Trust the Market’s Knee-Jerk Reaction to...,https://www.wsj.com/finance/stocks/dont-trust-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2461/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
77,trump,31
168,new,18
476,white,16
80,election,15
32,president,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
79,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...,146
140,2026-11-07-18-59-01 +0000,nypost,Trump offers UFC boss Dana White role of White...,https://nypost.com/2026/11/07/us-news/trump-of...,117
84,2026-11-08-00-08-30 +0000,wapo,Trump says he offered White House press secret...,https://www.washingtonpost.com/politics/2026/1...,114
77,2026-11-08-00-31-53 +0000,bbc,"'Dana, please take the job!' - Trump asks UFC ...",https://www.bbc.co.uk/news/articles/cvze47p9je...,101
166,2026-11-07-17-00-00 +0000,wsj,Young voters cited economic pressures as reaso...,https://www.wsj.com/politics/elections/the-new...,99


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
79,146,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...
219,69,2026-11-07-11-40-00 +0000,wsj,New Zealand’s ruling conservatives led by a sl...,https://www.wsj.com/world/oceania/new-zealands...
58,46,2026-11-08-02-00-00 +0000,wsj,A dominant midterm performance has given Democ...,https://www.wsj.com/politics/elections/the-new...
180,46,2026-11-07-16-00-28 +0000,chitri,Mary Miller vs Jennifer Todd: How Illinois’ 15...,https://www.chicagotribune.com/2026/11/07/mary...
101,43,2026-11-07-22-18-09 +0000,wapo,"Ex-military chief Gadi Eisenkot, opponent of N...",https://www.washingtonpost.com/world/2026/11/0...
113,40,2026-11-07-21-01-09 +0000,nypost,Shocking footage shows moment Krakow police sh...,https://nypost.com/2026/11/07/world-news/shock...
179,37,2026-11-07-16-09-00 +0000,wsj,Is AI a white-collar job killer? Strong hiring...,https://www.wsj.com/economy/jobs/is-ai-taking-...
97,32,2026-11-07-22-30-51 +0000,nypost,NY AG Letitia James lashes out at ‘drunk on po...,https://nypost.com/2026/11/07/us-news/ny-ag-le...
125,32,2026-11-07-20-12-26 +0000,nyt,"Father Calls Daughter, the Co-Pilot Killed in ...",https://www.nytimes.com/2026/11/07/us/florida-...
144,32,2026-11-07-18-49-11 +0000,nypost,Marco Rubio warns top staff against conflicts ...,https://nypost.com/2026/11/07/us-news/marco-ru...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
